In [17]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
spark = SparkSession.builder \
.appName("Read Parquet") \
    .config("spark.driver.memory", "12g")\
        .getOrCreate()

data = spark.read.parquet("training_data/combined_yellow_fhvhv.parquet")
data.show(20)

+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+
|      date|PULocationID|     total_revenue| log_total_revenue|lag_1_log_total_revenue|lag_7_log_total_revenue|lag_14_log_total_revenue|        driver_pay|    log_driver_pay|lag_1_log_driver_pay|lag_7_log_driver_pay|lag_14_log_driver_pay|
+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+
|2023-01-01|           7|1685.8999999999999| 7.430647803966266|                   NULL|                   NULL|                    NULL|135104.27000000005| 11.81380953132961|                NULL|                NULL|                 NULL|
|2023-01-01|          56|244.20999999999998|

# Bringing in the other features
We will bring in the external data as additional predictors for our models to gauge demand and competition with taxis.

In [18]:
from utils import add_stations
station_data = spark.read.csv('data/station_data.csv', header=True, inferSchema=True)
station_data.show(1)

+------------+----------+----------+--------+-------+--------------------+-------+-----+--------------+---------+-------------+--------------+---------------------+---------------------+---+--------------+--------------+---------+--------------------+
|GTFS Stop ID|Station ID|Complex ID|Division|   Line|           Stop Name|Borough|  CBD|Daytime Routes|Structure|GTFS Latitude|GTFS Longitude|North Direction Label|South Direction Label|ADA|ADA Northbound|ADA Southbound|ADA Notes|        Georeference|
+------------+----------+----------+--------+-------+--------------------+-------+-----+--------------+---------+-------------+--------------+---------------------+---------------------+---+--------------+--------------+---------+--------------------+
|         R01|         1|         1|     BMT|Astoria|Astoria-Ditmars Blvd|      Q|false|           N W| Elevated|    40.775036|    -73.912034|            Last Stop|            Manhattan|  0|             0|             0|     NULL|POINT (-73.912

In [19]:
data_1 = add_stations(station_data, data, spark)

/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/aydin-khan/Documents/coding/applied-data-science-projects/applied-ds-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_versi

In [20]:
data_1.show(20)

+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+------------+
|      date|PULocationID|     total_revenue| log_total_revenue|lag_1_log_total_revenue|lag_7_log_total_revenue|lag_14_log_total_revenue|        driver_pay|    log_driver_pay|lag_1_log_driver_pay|lag_7_log_driver_pay|lag_14_log_driver_pay|num_stations|
+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+------------+
|2023-01-01|           7|1685.8999999999999| 7.430647803966266|                   NULL|                   NULL|                    NULL|135104.27000000005| 11.81380953132961|                NULL|                NULL|                 NULL|      

In [21]:
from utils import add_unemployment_rate
data_2 = add_unemployment_rate(data_1, spark)
data_2.show()

+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+------------+----+-----+-----------------+
|      date|PULocationID|     total_revenue| log_total_revenue|lag_1_log_total_revenue|lag_7_log_total_revenue|lag_14_log_total_revenue|        driver_pay|    log_driver_pay|lag_1_log_driver_pay|lag_7_log_driver_pay|lag_14_log_driver_pay|num_stations|Year|Month|unemployment_rate|
+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+------------+----+-----+-----------------+
|2023-01-01|         177|              69.8| 4.259859000699674|                   NULL|                   NULL|                    NULL| 49111.98000000024|10

In [22]:
data_2.filter(data_2['unemployment_rate'].isNull()).show(20)

+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+------------+----+-----+-----------------+
|      date|PULocationID|     total_revenue| log_total_revenue|lag_1_log_total_revenue|lag_7_log_total_revenue|lag_14_log_total_revenue|        driver_pay|    log_driver_pay|lag_1_log_driver_pay|lag_7_log_driver_pay|lag_14_log_driver_pay|num_stations|Year|Month|unemployment_rate|
+----------+------------+------------------+------------------+-----------------------+-----------------------+------------------------+------------------+------------------+--------------------+--------------------+---------------------+------------+----+-----+-----------------+
|2025-10-07|          65|           2059.15| 7.630534074666685|      7.791130180094634|     7.7464346783265405|       7.390452996286758|58716.960000000036|10

In [23]:
from pyspark.sql import Window
# Get September and November rates for each PULocationID
sept = (
    data_2.filter(
        (F.col("Year") == 2025) & 
        (F.col("Month") == 9)
    )
    .select(
        "PULocationID",
        F.col("unemployment_rate").alias("sept_rate")
    )
)

nov = (
    data_2.filter(
        (F.col("Year") == 2025) & 
        (F.col("Month") == 11)
    )
    .select(
        "PULocationID",
        F.col("unemployment_rate").alias("nov_rate")
    )
)

# Join September and November rates back to the original dataframe
data_2 = (
    data_2
    .join(sept, on="PULocationID", how="left")
    .join(nov, on="PULocationID", how="left")
)

# Fill October 2025 NULLs using linear interpolation
data_2 = data_2.withColumn(
    "unemployment_rate",
    F.when(
        (F.col("Year") == 2025) &
        (F.col("Month") == 10) &
        F.col("unemployment_rate").isNull() &
        F.col("sept_rate").isNotNull() &
        F.col("nov_rate").isNotNull(),
        (F.col("sept_rate") + F.col("nov_rate")) / 2
    ).otherwise(F.col("unemployment_rate"))
)

# Remove helper columns
data_2 = data_2.drop("sept_rate", "nov_rate")

In [25]:
data_2.filter(data_2['unemployment_rate'].isNull()).count()

0

In [26]:
data_2.show()

+------------+----------+------------------+-----------------+-----------------------+-----------------------+------------------------+-----------------+------------------+--------------------+--------------------+---------------------+------------+----+-----+-----------------+
|PULocationID|      date|     total_revenue|log_total_revenue|lag_1_log_total_revenue|lag_7_log_total_revenue|lag_14_log_total_revenue|       driver_pay|    log_driver_pay|lag_1_log_driver_pay|lag_7_log_driver_pay|lag_14_log_driver_pay|num_stations|Year|Month|unemployment_rate|
+------------+----------+------------------+-----------------+-----------------------+-----------------------+------------------------+-----------------+------------------+--------------------+--------------------+---------------------+------------+----+-----+-----------------+
|         241|2023-01-01|60.400000000000006|4.117409835153097|                   NULL|                   NULL|                    NULL|41477.40000000006|10.6329280